<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drexelwireless/ece512-courseware/blob/main/notebooks/lectures/week01-cellular-geometry.ipynb)

# Week 1 — Cellular Geometry and Frequency Reuse
**ECE 512 Wireless Communications** · companion notebook to the Week 1 lecture

This notebook lets you experiment with the ideas from the slides:

1. Why hexagonal cells?
2. Cluster sizes $N = i^2 + ij + j^2$
3. Frequency reuse: tiling the plane with clusters
4. Finding co-channel cells ("move $i$, turn $60^\circ$, move $j$")
5. The reuse ratio $Q = D/R = \sqrt{3N}$
6. Friis free-space propagation

Run the setup cell first. In Colab: *Runtime → Run all*.

In [ ]:
# --- Setup: installs the course package in Google Colab (safe to re-run) ---
COURSEWARE_URL = "git+https://github.com/drexelwireless/ece512-courseware"
import importlib.util
if importlib.util.find_spec("ece512") is None:
    get_ipython().run_line_magic("pip", f"install -q {COURSEWARE_URL}")

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider
from ece512 import cells, propagation
from ece512.hexgrid import draw_cell, fill_cell, cell_axes, hex_vertices

## 1. Why hexagonal cells?
Only three regular polygons tile the plane without gaps: the triangle, the square, and the hexagon.
For a base station with an omnidirectional antenna the coverage contour is (ideally) a circle of radius $R$.
The hexagon is the tiling shape whose area comes closest to that circle, so fewer cells are needed to cover a region.

In [ ]:
shapes = {"triangle": 3, "square": 4, "hexagon": 6}
R = 1.0
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, (name, n) in zip(axes, shapes.items()):
    t = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), "--", color="gray")
    v = np.exp(2j * np.pi * np.arange(n + 1) / n)
    ax.fill(v.real, v.imag, alpha=0.3)
    area_ratio = 0.5 * n * np.sin(2 * np.pi / n) / np.pi   # polygon area / circle area
    ax.set_title(f"{name}: {100 * area_ratio:.1f}% of circle")
    ax.set_aspect("equal"); ax.axis("off")
plt.show()

## 2. Allowed cluster sizes
Clusters tessellate only when $N = i^2 + ij + j^2$ for non-negative integers $i, j$.

In [ ]:
allowed = sorted({cells.cluster_size(i, j) for i in range(6) for j in range(6)} - {0})
print("Allowed N:", [n for n in allowed if n <= 40])

In [ ]:
colors = plt.cm.tab20(np.linspace(0, 1, 20))
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (i, j) in zip(axes, [(1, 1), (2, 0), (2, 1), (2, 2)]):
    cell_axes(ax)
    cells.draw_cluster(ax, 0j, i, j, 1.0, colors=colors)
    ax.set_title(f"N = {cells.cluster_size(i, j)}  (i={i}, j={j})")
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.axis("off")
plt.show()

## 3. Frequency reuse: tiling the plane
Each letter is a disjoint set of channels. The central cluster is repeated around itself;
cells with the same letter reuse the same channels. Move the sliders to change $i$ and $j$.

In [ ]:
def show_reuse(i=2, j=1):
    n = cells.cluster_size(i, j)
    if n == 0:
        print("i and j cannot both be zero"); return
    ax = cell_axes(figsize=(7, 7))
    colors = plt.cm.tab20(np.linspace(0, 1, max(n, 2)))
    for k, offset in enumerate(np.concatenate([[0j], cells.cochannel_offsets(i, j, 1.0)])):
        centers = cells.cluster_centers(offset, i, j, 1.0)
        for g, c in enumerate(centers):
            fill_cell(ax, c, 1.0, colors[g % len(colors)], alpha=0.55 if k == 0 else 0.25)
            draw_cell(ax, c, 1.0, label=chr(65 + g) if g < 26 else str(g))
    ax.set_title(f"N = {n}: central cluster (dark) and its six co-channel neighbors")
    ax.axis("off"); plt.show()

interact(show_reuse, i=IntSlider(value=2, min=0, max=4), j=IntSlider(value=1, min=0, max=4));

## 4. Finding co-channel cells
From the home cell: move $i$ cells along a chain of hexagons, turn $60^\circ$ counter-clockwise, and move $j$ cells.
Repeating for all six starting directions gives the first tier of co-channel cells, each a distance $D = R\sqrt{3N}$ away.

In [ ]:
def show_cochannel(i=1, j=2):
    R = 1.0
    a1, a2 = cells.lattice_basis(R)
    ax = cell_axes(figsize=(7, 7))
    span = i + j + 1
    for m in range(-span - 1, span + 2):
        for n in range(-span - 1, span + 2):
            c = m * a1 + n * a2
            if abs(c) <= cells.reuse_distance(i, j, R) + 1.5 * R:
                draw_cell(ax, c, R, color="lightgray")
    fill_cell(ax, 0j, R, "tab:blue", alpha=0.6)
    rot = np.exp(1j * np.pi / 3)
    for k in range(6):
        d1, d2 = a1 * rot**k, a2 * rot**k
        corner = i * d1
        ax.annotate("", xy=(corner.real, corner.imag), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->", color="tab:green", lw=1.5))
        end = corner + j * d2
        ax.annotate("", xy=(end.real, end.imag), xytext=(corner.real, corner.imag),
                    arrowprops=dict(arrowstyle="->", color="tab:red", lw=1.5))
        fill_cell(ax, end, R, "tab:orange", alpha=0.6)
    D = cells.reuse_distance(i, j, R)
    ax.set_title(f"i={i} (green), j={j} (red): N={cells.cluster_size(i, j)}, D/R={D:.2f}")
    ax.axis("off"); plt.show()

interact(show_cochannel, i=IntSlider(value=1, min=0, max=4), j=IntSlider(value=2, min=0, max=4));

## 5. Reuse ratio and a preview of interference
The co-channel reuse ratio $Q = D/R = \sqrt{3N}$ grows with cluster size. In Week 2 we will show that, with
six equidistant first-tier interferers and path-loss exponent $\beta$, the signal-to-interference ratio is roughly
$\Lambda \approx Q^\beta / 6$: larger clusters mean less interference, but fewer channels per cell.

In [ ]:
print(f"{'N':>4} {'D/R':>6} {'SIR (beta=4)':>14} {'channels/cell (S=420)':>22}")
for n in [3, 4, 7, 9, 12, 13, 19, 21, 27]:
    q = np.sqrt(3 * n)
    print(f"{n:>4} {q:6.2f} {10 * np.log10(q**4 / 6):11.1f} dB {420 // n:>15}")

## 6. Friis free-space equation
$$P_{\mathrm{RX}}(d) = P_{\mathrm{TX}}\,\frac{G_{\mathrm{TX}}\,G_{\mathrm{RX}}}{L_{\mathrm{sys}}}\left(\frac{\lambda}{4\pi d}\right)^2$$
Received power falls 20 dB per decade of distance, and higher carrier frequencies (shorter $\lambda$) lose more power
for the same antenna gains.

In [ ]:
def show_friis(ptx_dbm=40.0, gains_dbi=10.0):
    d = np.logspace(1, 4, 200)
    fig, ax = plt.subplots(figsize=(7, 4))
    for f_mhz in [700, 1900, 3500, 28000]:
        p = propagation.friis_received_power_dbm(ptx_dbm, d, f_mhz * 1e6, gains_dbi, 0)
        ax.semilogx(d, p, label=f"{f_mhz / 1000:g} GHz")
    ax.set_xlabel("distance d (m)"); ax.set_ylabel(r"$P_{\mathrm{RX}}$ (dBm)")
    ax.grid(True, which="both", alpha=0.3); ax.legend(); plt.show()

interact(show_friis, ptx_dbm=FloatSlider(value=40, min=0, max=60, step=1), gains_dbi=FloatSlider(value=10, min=0, max=30, step=1));

## Try it yourself
* For $N = 7$, which cells in the first tier are closest to a user at the edge of the home cell? How does that
  change the "worst case" interference compared with a user at the cell center?
* Why is $N = 1$ (every cell uses every channel) attractive for capacity, and what problem does it create?
* At what distance does a 28 GHz link with 30 dBi of total antenna gain match a 700 MHz link with no gain?